<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Integrate Langfuse with Strands Box" sidebarTitle: "Strands Box" logo: "/images/integrations/strands-box_icon.svg" description: "Trace Strands agents running in Box with Langfuse through an operator-owned OpenTelemetry Collector, locally and on AgentCore Runtime." category: "Integrations" -->

# Integrate Langfuse with Strands Box

This guide sends native Strands agent traces through **Strands Box** to **Langfuse**, with Langfuse credentials kept in an operator-owned OpenTelemetry Collector.

> **What is Strands Box?** [Strands Box](https://strandsagents.com/docs/user-guide/box/) runs an agent with declared filesystem, network and credential access.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform for tracing and debugging AI applications.

<Frame>
![A Box-contained Strands agent in Langfuse, with two model calls and a README tool call](/images/cookbook/integration-strands-box/box-readme-trace.jpg)
</Frame>

The trace retains the original messages, tool results, token usage and Box run metadata. The screenshot shows a synthetic README task on AgentCore; it also preserves extra model text that failed the application's strict answer-format check.

<!-- CALLOUT_START type: "info" -->
Start with a working Box configuration for your Python Strands agent, including its interpreter, dependency and file grants, policy, model egress and operator-held model credential. Follow the [Box Python agent guide](https://strandsagents.com/docs/user-guide/box/guides/run-strands-harness/) first. This notebook adds observability to that configuration; it does not generate a containment policy.
<!-- CALLOUT_END -->

## How traces travel

```text
Contained Strands agent → Box loopback receiver
                          ↓
                  Operator-owned Collector
                  agent spans + Basic auth
                          ↓
                       Langfuse
```

Box supplies the workload's OTLP endpoint and stamps relayed resources with its run ID. The Collector selects agent spans, adds Langfuse authentication and forwards traces. Local Box policy/control records can stay in the operator's record file.

<!-- CALLOUT_START type: "warning" -->
This recipe was qualified with Box revision `e4ca8b80359b2213d684de31cd6ef32e0d61478a`, Strands 1.63.0 and Collector Contrib 0.164.0. The tested native Box exporter prefixes Authorization secrets with `Bearer ` and cannot add the v4 header. Langfuse requires Basic authentication, so use the Collector route below for this version. Langfuse accepts both OTLP/HTTP JSON and protobuf; protobuf support is not the missing piece. Recheck compatibility when upgrading Box.
<!-- CALLOUT_END -->

<!-- STEPS_START -->
## Step 1: Install dependencies

Use Python 3.11 or later for this notebook. Install the same Strands version into the environment that your Box workload uses; installing it in the operator notebook alone does not grant the contained process access to it.

In [ ]:
%pip install -U langfuse "strands-agents[otel]==1.63.0" tomli-w

## Step 2: Set up the operator environment

Get your Langfuse keys from the project settings in [Langfuse Cloud](https://langfuse.com/cloud) or set up [self-hosting](https://langfuse.com/self-hosting).

In [ ]:
import os

# Get keys for your project from the project settings page: https://langfuse.com/cloud
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...");
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"); # 🇪🇺 EU region (API host)
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

os.environ.setdefault("BOX_BIN", "/absolute/path/to/box");
os.environ.setdefault("BOX_CONFIG_PATH", "/absolute/path/to/your/box.toml");
os.environ.setdefault("OTELCOL_BIN", "/absolute/path/to/otelcol-contrib");

With the environment variables set, initialize the Langfuse client. `get_client()` picks up the env vars above and returns a client bound to your project.


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## Step 3: Initialize tracing inside the contained agent

Add this code to the existing Python entrypoint that your Box configuration launches. The `agent` and `prompt` below are your application's existing objects. Do not run the agent in the operator notebook process.

```python
# In the Python entrypoint launched by Box, around your existing agent call:
from strands.telemetry import StrandsTelemetry

telemetry = StrandsTelemetry().setup_otlp_exporter(timeout=5)
try:
    result = agent(prompt)  # Your existing Strands Agent and task.
finally:
    if not telemetry.tracer_provider.force_flush(timeout_millis=10000):
        raise RuntimeError("The agent trace exporter did not flush")
    telemetry.tracer_provider.shutdown()
```

Do not initialize a Langfuse SDK client or configure a Langfuse OTLP destination inside the contained agent. Box owns `OTEL_EXPORTER_OTLP_*` and supplies its loopback receiver. Keep the agent's declared environment free of Langfuse keys and Runtime-role credentials. The operator client above is only an authentication check.

## Step 4: Start an operator-owned Collector

Install [OpenTelemetry Collector Contrib](https://github.com/open-telemetry/opentelemetry-collector-releases/releases/tag/v0.164.0) 0.164.0 for your operating system and architecture. This recipe requires the OTLP receiver/exporter, filter and batch processors, health-check extension, and nop exporter.

The following helper binds loopback ports and waits for readiness. It uses a clean Collector environment. The Collector configuration stays in an operator-only temporary directory that is removed when the helper stops. The Collector log is kept, mode 0600, and the helper prints its path. The trace filter drops Box records from this destination; the logs and metrics pipelines terminate locally. It does not rewrite Strands messages, usage or parent IDs.

In [ ]:
"""Notebook helper: launch an operator-owned Collector for an existing Box workload."""
from contextlib import contextmanager
from pathlib import Path
import base64
import os
import socket
import subprocess
import tempfile
import time
from urllib.request import urlopen

COLLECTOR_YAML = r'''receivers:
  otlp:
    protocols:
      http:
        endpoint: 127.0.0.1:${env:BOX_COLLECTOR_PORT}
processors:
  filter/agent_only:
    error_mode: propagate
    trace_conditions:
      - 'resource.attributes["strands.box.source"] != "agent"'
  batch:
    timeout: 1s
exporters:
  otlp_http/langfuse:
    endpoint: ${env:LANGFUSE_OTLP_ENDPOINT}
    encoding: json
    compression: none
    headers:
      Authorization: ${env:LANGFUSE_OTLP_AUTH}
      x-langfuse-ingestion-version: "4"
    timeout: 10s
    retry_on_failure:
      enabled: true
      initial_interval: 1s
      max_interval: 5s
      max_elapsed_time: 30s
    sending_queue:
      enabled: true
      queue_size: 100
  nop:
extensions:
  health_check:
    endpoint: 127.0.0.1:${env:BOX_HEALTH_PORT}
service:
  extensions: [health_check]
  pipelines:
    traces:
      receivers: [otlp]
      processors: [filter/agent_only, batch]
      exporters: [otlp_http/langfuse]
    logs:
      receivers: [otlp]
      exporters: [nop]
    metrics:
      receivers: [otlp]
      exporters: [nop]
'''


def free_ports():
    sockets = [socket.socket() for _ in range(2)]
    try:
        for item in sockets:
            item.bind(("127.0.0.1", 0))
        return [item.getsockname()[1] for item in sockets]
    finally:
        for item in sockets:
            item.close()


def collector_log_tail(path, limit=40):
    """Return a short log excerpt with credential-bearing lines removed."""
    try:
        text = path.read_text(errors="replace")
    except OSError:
        return ""
    kept = []
    for line in text.splitlines():
        lowered = line.lower()
        if any(
            token in lowered
            for token in (
                "authorization",
                "langfuse_otlp_auth",
                "langfuse_secret_key",
                "sk-lf-",
                "pk-lf-",
            )
        ):
            kept.append("[redacted line]")
        else:
            kept.append(line)
    return "\n".join(kept[-limit:])


@contextmanager
def start_collector(*, destination=None, authorization=None):
    """Keep credentials in the operator's Collector process, outside the agent."""
    port, health_port = free_ports()
    if authorization is None:
        encoded = base64.b64encode(
            f"{os.environ['LANGFUSE_PUBLIC_KEY']}:{os.environ['LANGFUSE_SECRET_KEY']}".encode()
        ).decode()
        authorization = f"Basic {encoded}"
    destination = destination or os.environ["LANGFUSE_BASE_URL"].rstrip("/") + "/api/public/otel"
    # mkstemp creates the log as 0600 and does not delete it when this helper exits.
    descriptor, log_name = tempfile.mkstemp(prefix="box-langfuse-collector-", suffix=".log")
    os.close(descriptor)
    log_path = Path(log_name)
    startup_error = None
    try:
        with tempfile.TemporaryDirectory(prefix="box-langfuse-") as directory:
            config = Path(directory) / "collector.yaml"
            config.write_text(COLLECTOR_YAML)
            child_env = {key: os.environ[key] for key in ("HOME", "PATH", "TMPDIR") if key in os.environ}
            child_env.update({
                "BOX_COLLECTOR_PORT": str(port),
                "BOX_HEALTH_PORT": str(health_port),
                "LANGFUSE_OTLP_ENDPOINT": destination,
                "LANGFUSE_OTLP_AUTH": authorization,
            })
            # Keep the log after this helper exits. Do not print its contents.
            with log_path.open("w") as log:
                process = subprocess.Popen(
                    [os.environ["OTELCOL_BIN"], "--config", str(config)],
                    env=child_env, stdout=log, stderr=log,
                )
                try:
                    deadline = time.monotonic() + 30
                    while True:
                        if process.poll() is not None:
                            startup_error = RuntimeError("Collector exited before becoming ready")
                            break
                        try:
                            with urlopen(f"http://127.0.0.1:{health_port}/", timeout=0.2) as response:
                                if response.status == 200:
                                    break
                        except OSError:
                            pass
                        if time.monotonic() >= deadline:
                            startup_error = TimeoutError("Collector did not become ready")
                            break
                        time.sleep(0.1)
                    if startup_error is None:
                        yield f"http://127.0.0.1:{port}"
                finally:
                    if process.poll() is None:
                        process.terminate()
                    try:
                        process.wait(timeout=45)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                    log.flush()
    finally:
        print(f"Collector log: {log_path}")
    if startup_error is not None:
        tail = collector_log_tail(log_path)
        message = f"{startup_error}\nCollector log: {log_path}"
        if tail:
            message += f"\n{tail}"
        raise type(startup_error)(message) from startup_error



## Step 5: Connect Box and run your existing workload

The helper below creates a temporary copy beside your base configuration, preserving relative paths and existing containment grants. It adds the Collector target, retains the default local record file when no targets were declared, and passes only explicitly referenced egress secrets to the Box process. Your existing operator model credential must already be available; the notebook does not create it.

For Linux, complete Box's documented platform prerequisites and set `EXPERIMENTAL_LINUX_SUPPORT=TRUE` in the operator environment when required by the tested version. Run as a non-root operator. The two-port allocation is suitable for this single-run example; production supervisors should manage endpoints and concurrency.

In [ ]:
"""Add telemetry to a working Box config without changing its containment grants."""
from pathlib import Path
import os
import subprocess
import tempfile
import tomllib
import tomli_w


def run_box_with_collector(config_path, collector_endpoint):
    config_path = Path(config_path).resolve()
    config = tomllib.loads(config_path.read_text())
    agent_env = config["agent"].setdefault("env", {})
    if any(key.startswith(("LANGFUSE_", "OTEL_EXPORTER_OTLP_")) for key in agent_env):
        raise ValueError("Remove Langfuse credentials and OTLP endpoints/headers from agent.env")
    agent_env["OTEL_TRACES_EXPORTER"] = "otlp"
    agent_env.setdefault("OTEL_SERVICE_NAME", "strands-box-example")
    targets = config.setdefault("telemetry", {})
    if not targets:
        # An explicit target replaces Box's default file target. Retain that file.
        targets["local"] = {
            "kind": "file",
            "destination": str(Path(config["box_dir"]).expanduser() / "private/telemetry/records.jsonl"),
        }
    if "langfuse_collector" in targets:
        raise ValueError("Use a base config without a langfuse_collector target")
    targets["langfuse_collector"] = {
        "kind": "otlp", "destination": collector_endpoint, "include": ["trace"],
    }
    box_env = {key: os.environ[key] for key in ("HOME", "PATH", "TMPDIR", "EXPERIMENTAL_LINUX_SUPPORT") if key in os.environ}
    missing = []
    for egress in config.get("egress", {}).values():
        reference = egress.get("secret", {}).get("ref", "")
        if reference.startswith("env://"):
            name = reference.removeprefix("env://")
            if name not in os.environ:
                missing.append(name)
            else:
                box_env[name] = os.environ[name]
    if missing:
        raise KeyError(
            "Missing operator environment variables for Box egress: " + ", ".join(missing)
        )
    # Preserve the config directory so relative policy/command paths still resolve.
    # Create the overlay only after egress secrets resolve, and always remove it.
    with tempfile.NamedTemporaryFile(dir=config_path.parent, suffix=".toml", delete=False) as file:
        overlay = Path(file.name)
    try:
        overlay.write_bytes(tomli_w.dumps(config).encode())
        result = subprocess.run(
            [os.environ["BOX_BIN"], "run", "--config", str(overlay)],
            env=box_env, timeout=120, check=True,
        )
        return result.returncode
    finally:
        overlay.unlink(missing_ok=True)

with start_collector() as collector_endpoint:
    exit_code = run_box_with_collector(os.environ["BOX_CONFIG_PATH"], collector_endpoint)
print("Box workload exit code:", exit_code)


## Step 6: View traces in Langfuse

Open [your project's tracing view](https://cloud.langfuse.com/project/~/traces). Inspect the agent, model calls and tools, then open Metadata to inspect Box provenance. The exact number of observations depends on the workload.

<!-- STEPS_END -->

## Run the same pattern on AgentCore Runtime [#agentcore]

Keep the three components in one Runtime session: the operator entrypoint, Box's contained agent process, and the operator-owned Collector. Our reference deployment used Runtime V2, a non-root operator, Linux ARM64 and Python 3.13. Build native dependencies for the selected Runtime architecture and Python version.

Use the [official Box AgentCore deployment guide](https://strandsagents.com/docs/user-guide/box/guides/deploy-to-agentcore-runtime/) for platform setup and the [Langfuse AgentCore guide](https://langfuse.com/integrations/frameworks/amazon-agentcore) for external telemetry configuration. The Box guide's Python 3.12 shipment example is a different workload from the README tracing example shown here.

1. Package the Box binary and required helpers, your contained agent, Python dependencies, policy/configuration, and the Collector binary/configuration. Keep credentials out of the package.
2. Start the Runtime's operator as a non-root user. Retain model permissions in its execution role and resolve the model credential outside the agent. Our Bedrock route used a short-lived bearer token referenced by Box egress; the contained workload received Box's replacement credential.
3. Set `DISABLE_ADOT_OBSERVABILITY=true` for the external telemetry setup used here. Supply Langfuse settings to the operator/Collector, with network access to the correct Langfuse regional host. Do not copy Runtime-role credentials, Langfuse headers or operator OTLP variables into `[agent.env]`.
4. Start the Collector and check readiness before invoking Box. Keep it running until agent flush and Collector shutdown complete. Construct fresh Box state for each isolated task where your application requires it.
5. Test a fresh Runtime session and a repeated invocation. Verify the DEFAULT endpoint points at the new version before judging a test result.

<!-- CALLOUT_START type: "warning" -->
The tested Collector binary is about 168 MB. Its initial read/startup was slow in our S3 ZIP deployment. An invocation-time sequential read and checksum verification avoided the observed readiness failure. Measure cold-session behavior and restored sessions with your package; a warm local process does not qualify Runtime startup. This guide does not establish the underlying platform cause.
<!-- CALLOUT_END -->

### Include caller context and the outer request [#request-context]

The inner `invoke_agent` duration covers the agent workflow. Add an operator span if you also need package preparation, Collector startup, Box launch, validation, and the Runtime's final error in the same trace.

Use an operator-owned tracer/exporter with Langfuse Basic authentication. Extract the caller's W3C `traceparent`/`tracestate`, start `invoke-runtime`, and inject that current context into the explicitly declared agent environment:

```python
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

propagator = TraceContextTextMapPropagator()
# In your Runtime handler; `tracer` belongs to the operator's provider.
parent = propagator.extract(payload.get("trace_context", {}))
with tracer.start_as_current_span("invoke-runtime", context=parent) as runtime_span:
    carrier = {}
    propagator.inject(carrier)
    agent_env.update({
        "TRACEPARENT": carrier["traceparent"],
        "TRACESTATE": carrier.get("tracestate", ""),
    })
    # Launch Box with this agent_env, record the response/error on runtime_span,
    # and end the span after request validation and cleanup.
```

In the contained Python entrypoint, attach that context before calling the agent:

```python
import os
from opentelemetry import context
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

parent = TraceContextTextMapPropagator().extract({
    "traceparent": os.environ.get("TRACEPARENT", ""),
    "tracestate": os.environ.get("TRACESTATE", ""),
})
token = context.attach(parent)
try:
    result = agent(prompt)
finally:
    context.detach(token)
    # Force-flush the provider as in the native agent setup above.
```

W3C trace context contains correlation identifiers, not credentials or authorization. Validate the caller payload according to your application.

<!-- CALLOUT_START type: "info" -->
This context pattern passed local ARM64 tests using the actual AgentCore HTTP application and Box with fixed model responses. It also passed API readback for two real AgentCore invocations: eight observations without an upstream caller, and nine with explicitly forwarded caller context. The caller in the latter test was a manually timed console-test driver, not automatic AWS client instrumentation. Both cloud tasks returned HTTP 500 for the strict answer-format check, and their complete traces and request errors arrived. The HTTP-200 path was qualified locally.
<!-- CALLOUT_END -->

<!-- CALLOUT_START type: "warning" -->
The operator span begins after the Runtime framework parses the JSON request. It covers preparation and failures inside the application handler; it does not cover invalid-JSON rejection, platform authentication, or process startup before the handler. Add server middleware or platform telemetry for those boundaries.
<!-- CALLOUT_END -->

## Check delivery and representation [#verification]

Use a known trace ID and [query observations through the API](https://langfuse.com/docs/api-and-data-platform/features/query-via-sdk). Include input/output, model, usage and expanded resource metadata. Poll within a bounded interval while ingestion completes; a single immediate empty response is inconclusive.

- Check all expected observation IDs and parent links. For the two-model-call README example, expect one AGENT, two SPANs, two GENERATIONs and one TOOL.
- Match tool-use and tool-result IDs, and confirm the second generation received the actual tool result.
- Sum usage on GENERATION observations. Do not add the agent's aggregate usage again.
- Check `strands.box.source=agent`, `strands.box.run.id`, and `strands.box.name` under resource metadata. These are Box-stamped correlation attributes, not a cryptographic attestation.
- Open the Collector log path printed when the helper stops. Check export errors as well as counters, and read back the observations. Leave credential-bearing lines out of anything you share. HTTP acceptance or a sent-span counter alone does not prove that every span was ingested.

<!-- CALLOUT_START type: "warning" -->
Tracing success and task success are separate. The screenshot's six agent observations arrived even though the Runtime rejected extra answer text with HTTP 500. Preserve that raw output and record the request outcome separately. The queue in this recipe is in memory and retries expire after 30 seconds; it does not provide durable or exactly-once delivery.
<!-- CALLOUT_END -->

## References

- [Box telemetry guide](https://strandsagents.com/docs/user-guide/box/guides/record-telemetry/)
- [Box configuration reference](https://strandsagents.com/docs/user-guide/box/reference/configuration/)
- [Strands integration](https://langfuse.com/integrations/frameworks/strands-agents)
- [Langfuse native OpenTelemetry integration](https://langfuse.com/integrations/native/opentelemetry)


<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->